This notebook creates a training model for one month (April 2026) of NYC TLC green taxi trip data
(Parquet).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
import numpy as np


In [7]:
df = pd.read_parquet('../data/green_tripdata_2026-04.parquet')

df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2026-04-01 00:29:09,2026-04-01 00:54:48,N,1.0,66,48,1.0,6.56,31.0,...,0.5,11.10,0.0,NaN,1.0,48.10,1.0,1.0,2.75,0.75
1,2,2026-04-01 00:19:13,2026-04-01 00:19:20,N,5.0,129,129,1.0,0.00,26.0,...,0.0,0.00,0.0,NaN,1.0,27.00,1.0,2.0,0.00,0.00
2,2,2026-04-01 00:40:24,2026-04-01 00:47:31,N,1.0,74,141,1.0,2.56,12.1,...,0.5,3.47,0.0,NaN,1.0,20.82,1.0,1.0,2.75,0.00
3,2,2026-04-01 00:27:31,2026-04-01 00:35:33,N,1.0,244,116,1.0,1.48,10.0,...,0.5,0.00,0.0,NaN,1.0,12.50,2.0,1.0,0.00,0.00
4,2,2026-04-01 00:51:45,2026-04-01 00:58:33,N,1.0,243,235,1.0,0.93,8.6,...,0.5,0.01,0.0,NaN,1.0,11.11,1.0,1.0,0.00,0.00


In [8]:
print (df.columns)

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'cbd_congestion_fee'],
      dtype='str')


In [13]:
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)
df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
df['duration'] = df['duration'].dt.total_seconds() / 60

# Filter outliers (highly recommended for linear models)
# Standard practice is to keep trips between 1 and 60 minutes
df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

# Engineer the PU_DO feature (Pickup/Dropoff pair)
# We convert them to strings so DictVectorizer treats them as categories, not numbers
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)

# 4. Define your final features for the model
categorical = ['PU_DO']
numerical = ['trip_distance']

# Check your work
df[['duration', 'PU_DO', 'trip_distance']].head()

,duration,PU_DO,trip_distance
0,25.650000,66_48,6.56
2,7.116667,74_141,2.56
3,8.033333,244_116,1.48
4,6.800000,243_235,0.93
5,18.833333,75_235,6.05


In [14]:
# 1. Splitting data for training/testing 80% training data, random seed 42
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)


categorical = ['PU_DO']
numerical = ['trip_distance']

# 2. DictVectorizer requires lists of dictionaries
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
test_dicts = df_test[categorical + numerical].to_dict(orient='records')

# 3. Fit the Vectorizer and transform the data
dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)
X_test = dv.transform(test_dicts)

# Extract the target variable
y_train = df_train['duration'].values
y_val = df_test['duration'].values

# 4. Train the baseline model
lr = LinearRegression()
lr.fit(X_train, y_train)

# 5. Evaluate on the validation set
y_pred = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)

print(f"Validation RMSE: {rmse:.4f} minutes")
print(f"Validation MAE: {mae:.4f} minutes")

Validation RMSE: 6.7983 minutes
Validation MAE: 4.3995 minutes


In [ ]:
import pickle
import os

# Save both the vectorizer and the model as a tuple
os.makedirs('../models', exist_ok=True)
with open('../models/baseline.pkl', 'wb') as f:
    pickle.dump((dv, lr), f)


In [22]:
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
with open('../reports/module-1.md', 'a') as f:
    f.write(f"\n### Run: {timestamp}\n")
    f.write(f"- **RMSE:** {rmse:.4f}\n")
    f.write(f"- **MAE:** {mae:.4f}\n")
    f.write("---\n")

print("Metrics appended to reports/module-1.md")

Metrics appended to reports/module-1.md
